# Model Tuning & Deployment

After building a baseline model, **hyperparameter tuning** squeezes out better performance.
Then **saving the model** lets you reuse it without retraining every time.

## What's Covered

| Topic | Tool | Purpose |
|---|---|---|
| **Grid Search** | `GridSearchCV` | Try every combination of hyperparameters exhaustively |
| **Random Search** | `RandomizedSearchCV` | Sample random combinations — faster for large grids |
| **Cross-Validation** | `cross_val_score`, `StratifiedKFold` | More reliable evaluation than a single train/test split |
| **Pipeline** | `Pipeline`, `ColumnTransformer` | Chain preprocessing + model into one reusable object |
| **Model Saving** | `joblib`, `pickle` | Persist trained model to disk, reload for inference later |

## What Are Hyperparameters?

- **Model weights** (e.g. linear regression slopes) → learned automatically from data by `.fit()`
- **Hyperparameters** (e.g. `max_depth`, `n_neighbors`, `C`) → **you must set them before training**
- Choosing the right hyperparameter values can significantly improve accuracy
- Finding them manually by trial-and-error is tedious → `GridSearchCV` / `RandomizedSearchCV` automates this

In [ ]:
## ── MODEL SAVING & LOADING ───────────────────────────────────────────────────

import joblib
import pickle
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train a simple pipeline to demonstrate saving and loading
X, y = make_classification(n_samples=300, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  DecisionTreeClassifier(max_depth=5, random_state=42)),
])
pipeline.fit(X_train, y_train)

original_acc = accuracy_score(y_test, pipeline.predict(X_test))
original_preds = pipeline.predict(X_test)
print(f'Trained pipeline accuracy: {original_acc:.4f}')

# ── SAVING WITH JOBLIB ─────────────────────────────────────────────────────────
# joblib is the RECOMMENDED way to save sklearn models
# Faster than pickle for objects that contain large numpy arrays (which sklearn models do)
# Supports optional compression: joblib.dump(model, path, compress=3)

path_joblib = 'student_model.joblib'
joblib.dump(pipeline, path_joblib)  # serialise pipeline to disk

file_size = os.path.getsize(path_joblib)
print(f'\n=== Saved with joblib ===')
print(f'File: {path_joblib}  ({file_size} bytes on disk)')

# Reload — creates an exact copy of the original Python object
pipeline_jl = joblib.load(path_joblib)
jl_acc = accuracy_score(y_test, pipeline_jl.predict(X_test))
preds_match = np.array_equal(original_preds, pipeline_jl.predict(X_test))
print(f'Reloaded accuracy : {jl_acc:.4f}  (same as original: {jl_acc == original_acc})')
print(f'Predictions match : {preds_match}  (every prediction is identical)')

# ── SAVING WITH PICKLE ─────────────────────────────────────────────────────────
# pickle is Python's built-in serialization — works for ANY Python object
# Slightly slower than joblib for large arrays, but perfectly fine for small models
# 'wb' = write binary mode (required for pickle)
# 'rb' = read binary mode

path_pickle = 'student_model.pkl'
with open(path_pickle, 'wb') as f:
    pickle.dump(pipeline, f)

file_size_pkl = os.path.getsize(path_pickle)
print(f'\n=== Saved with pickle ===')
print(f'File: {path_pickle}  ({file_size_pkl} bytes on disk)')

with open(path_pickle, 'rb') as f:
    pipeline_pkl = pickle.load(f)

pkl_acc = accuracy_score(y_test, pipeline_pkl.predict(X_test))
print(f'Reloaded accuracy : {pkl_acc:.4f}')

# ── JOBLIB VS PICKLE ───────────────────────────────────────────────────────────
print('\n=== joblib vs pickle ===')
comparison = pd.DataFrame({
    'Feature':         ['Speed (large models)', 'Compression', 'sklearn support',  'Best for'],
    'joblib':          ['Faster',              'Built-in',    'Recommended',       'sklearn pipelines / large arrays'],
    'pickle':          ['Slower',              'None',        'Works fine',        'Small models / any Python object'],
})
print(comparison.to_string(index=False))

print('\nRecommendation: use joblib for sklearn models, pickle for general Python objects')

# ── REAL-WORLD PATTERN ────────────────────────────────────────────────────────
print('''
=== Real-World Usage Pattern ===

── TRAINING SCRIPT (train_model.py) ──────────────────────────────────────────
  pipeline = Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier())])
  pipeline.fit(X_train, y_train)
  joblib.dump(pipeline, 'model_v1.joblib')          # save to disk
  print('Model saved')

── INFERENCE / DEPLOYMENT SCRIPT (predict.py) ────────────────────────────────
  pipeline = joblib.load('model_v1.joblib')          # load ONCE at startup
  new_data  = pd.read_csv('new_students.csv')
  preds     = pipeline.predict(new_data)             # preprocessing is automatic
  print(preds)

Key point: because the Pipeline includes the scaler, the loaded model handles
raw (unscaled) data automatically — no manual preprocessing step needed.
''')

# Cleanup demo files
os.remove(path_joblib)
os.remove(path_pickle)
print('Demo files cleaned up.')

# ── PHASE 6 COMPLETE SUMMARY ──────────────────────────────────────────────────
print('''
=== Phase 6 Summary ===

GridSearchCV:
  → Tries every combination in param_grid (exhaustive)
  → Best for small grids (< ~100 combinations)
  → grid_search.best_params_   → winning hyperparameters
  → grid_search.best_estimator_ → already-fitted best model

RandomizedSearchCV:
  → Samples n_iter random combinations (approximate but fast)
  → Best for large grids or continuous distributions
  → Works the same API as GridSearchCV

StratifiedKFold + cross_val_score:
  → K-fold CV preserves class ratio in each fold
  → More reliable accuracy estimate than a single train/test split

Pipeline:
  → Pipeline([('step1', transformer), ('step2', model)])
  → .fit(X_train) fits every step; .predict(X_test) applies every step
  → Prevents data leakage, works with GridSearchCV via step__param syntax

ColumnTransformer:
  → Apply different pipelines to different columns simultaneously
  → preprocessor = ColumnTransformer([('num', num_pipe, num_cols),
                                       ('cat', cat_pipe, cat_cols)])

Model Saving:
  → joblib.dump(model, 'file.joblib')   /  joblib.load('file.joblib')
  → pickle.dump(model, open('file.pkl','wb'))  /  pickle.load(open('file.pkl','rb'))
''')